# Paragraph and report encoding: MiniLM

Adapted from the supplied original workflow. Outputs and personal paths have been removed. Read `docs/PREPARING_DATA.md` and configure `configs/preparation.local.json` first. Run cells in order. This adaptation has not been validated by a complete raw-to-neural rerun.


# MiniLM: encode the full accepted 10-K Item 1 paragraph corpus on Windows

English notebook, independent checkpoints, full-corpus defaults, no outcome-model fitting.

**Encoding unit:** one complete retained Item 1 paragraph, including paragraphs with zero competition mentions. Long paragraphs are split into token windows and recombined without dropping tokens. Exactly identical texts are encoded once, but every paragraph ID remains linked to its report. This is a document representation, not a claim that the embedding removes causal confounding.

**Model:** `sentence-transformers/all-MiniLM-L6-v2`. Saved paragraph dimension: **384**. The Llama notebook uses an embedding-trained Llama 3.2 1B derivative, not a chat model. MiniLM uses 256-token sequences; Llama uses 512-token sequences for throughput; OpenAI uses up to 8,191-token windows. The comparison is between these encoding pipelines, not model architecture alone.

All notebooks share the same frozen manifest, paragraph IDs, exact-text hashes and conservative navigation cleaning. The manifest includes accepted reports only, not sources rejected for missing Item 1, boundary review or extraction errors. Existing `review_required` flags remain available and are not automatically cleared. Navigation cleaning does not repair Item 1 boundaries.

**Run order:** install dependencies once; run MiniLM through Step 3 first. After `Shared corpus ready`, you may run the three notebooks together in separate kernels. Defaults: MiniLM CPU, Llama CUDA GPU, OpenAI API. Two local GPU runs are blocked from competing for the same GPU. All SQLite/vector outputs are local, outside Dropbox. Do not change settings or source files during a run.

**Potentially faster on a single GPU:** change MiniLM `DEVICE_REQUEST` to `cuda`, run MiniLM Step 4 to completion, then run Llama Step 4; OpenAI can run alongside either. Forcing both local models onto one GPU is not assumed to improve total throughput. The parallel CPU/GPU default trades some MiniLM speed for immediate concurrent execution. Measure the actual paragraphs/second on your machine.

**Resume:** after interruption, restart the kernel and run Steps 2–5. Already committed paragraphs are skipped. Installation is not needed again. A new model revision, package version, device or encoding setting creates a separate checkpoint instead of mixing vectors. The original caches, Excel exports and existing embeddings are read-only inputs.


## Step 1 — Environment (one time only)

Use a dedicated Python 3.10+ environment. If the existing Windows environment already has working CUDA PyTorch, reuse it; the cells below do not reinstall PyTorch. Install missing packages **before** starting the other two notebooks. Restart the kernel after installing. Do not install/upgrade shared packages while any run is active.

If `torch.cuda.is_available()` is false in Llama, install the Windows/CUDA PyTorch build matching your driver using [the official selector](https://pytorch.org/get-started/locally/). This notebook uses PyTorch/Hugging Face, not a Linux-only vLLM or NVIDIA NIM server.


In [ ]:
# Uncomment only if these dependencies are missing. Then restart this kernel.
# %pip install "numpy>=1.26,<3" "tqdm>=4.66" "filelock>=3.13" "transformers>=4.44,<6" "huggingface_hub>=0.28" "safetensors>=0.4" "openai>=1.50,<3" "tiktoken>=0.7"

import sys, importlib.util
assert sys.version_info >= (3, 10), "Use a Python 3.10+ kernel."
required = ['numpy', 'tqdm', 'filelock']
missing = [name for name in required if importlib.util.find_spec(name) is None]
assert not missing, f"Install missing packages and restart: {missing}"
print("Python:", sys.version.split()[0], "| kernel:", sys.executable)


## Step 2 — Configuration

Set the extraction input path in the preparation configuration. It must contain `all_years/cache_manifest.jsonl` and **all referenced gzip caches**. On Dropbox choose “Make available offline” for the recovered folder. Copying only the Excel parts is insufficient. Mac cache paths are resolved against the Windows folder automatically; a missing or ambiguous cache stops corpus preparation.

Outputs default to `the configured embedding_output_root`. Keep enough free space for the shared text database, three vector files and model downloads. Step 3 prints the vector-file estimate. Paragraph vectors use float16 to reduce disk space; report means are accumulated in float64 and saved in float32. Retain `corpus.sqlite` to recover text by paragraph ID.


In [ ]:
from pathlib import Path
import os

from pathlib import Path
import sys
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "project_config.py").is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("Start Jupyter from the repository root or notebooks directory.")
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))
from preparation_config import load_preparation_config
PREP = load_preparation_config(REPO_ROOT)

INPUT_ROOT = PREP["extraction_dir"]
DATA_DIR = INPUT_ROOT / "all_years" if (INPUT_ROOT / "all_years").is_dir() else INPUT_ROOT
OUTPUT_ROOT = PREP["embedding_output_root"]
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_LABEL = 'minilm'
MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
DIMENSION = 384
BUILD_CORPUS = True  # Only MiniLM builds the corpus. The others open the completed corpus.
REMOVE_EXACT_NAVIGATION = True  # Identical setting required in all three notebooks.
PARAGRAPHS_PER_COMMIT = 128
CPU_THREADS = min(4, max(1, (os.cpu_count() or 2) // 3))

DEVICE_REQUEST = os.environ.get('ENCODING_DEVICE', 'cpu')  # MiniLM CPU + Llama CUDA is the default concurrent configuration.
IS_LLAMA = False
MAX_SEQUENCE_LENGTH = 256
BATCH_SIZE = 32  # CUDA OOM automatically halves this down to 1.

assert DATA_DIR.is_dir(), f"Input folder missing: {DATA_DIR}"
assert (DATA_DIR / "cache_manifest.jsonl").is_file(), f"Manifest missing under {DATA_DIR}"
print("Input:", DATA_DIR)
print("Output:", OUTPUT_ROOT)


## Step 3 — Shared corpus and alignment

Run this step first in MiniLM. It reads each accepted report cache once; this initial full scan can take time. It checkpoints one report at a time, so interrupted preparation resumes safely. The other notebooks only read the finished corpus.

Only whole-paragraph exact navigation labels are removed: `Index`, `Table of contents`, `Back/Return to table of contents`, `Back/Return/Go to top`, and `Back/Return to index`, allowing case/spacing/punctuation variants. Business headings and substantive text are retained. Each report's sentence count and competition sentence share are recomputed over retained paragraphs using the existing sentence labels. No new sentiment or LDA model is fitted. If this cleaning differs from your older analysis table, use the newly exported competition intensity and paragraph counts after joining by `doc_id`.

The unique-text vector row is fixed as `uid - 1` for every model. Never pool repeated sentence rows as if they were separate paragraphs.


In [ ]:
from pathlib import Path
import os, sys, json, gzip, csv, hashlib, sqlite3, time, uuid, re, unicodedata
from collections import Counter, OrderedDict
from contextlib import contextmanager
import numpy as np
from tqdm.auto import tqdm
from filelock import FileLock

ENGINE_VERSION = 'paragraph_three_encoders_v1'
NAV_LABELS = {'index','table of contents','back to table of contents',
    'return to table of contents','back to top','return to top','go to top',
    'back to index','return to index'}

def digest(s):
    return hashlib.sha256(s.encode('utf-8')).hexdigest()

def file_sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda: f.read(8*1024*1024), b''): h.update(b)
    return h.hexdigest()

def atomic_json(path, obj):
    path = Path(path)
    tmp = path.with_name(path.name + '.' + uuid.uuid4().hex + '.tmp')
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding='utf-8')
    os.replace(tmp, path)

def is_navigation(text):
    s = unicodedata.normalize('NFKC', text).casefold()
    s = re.sub(r'\s+', ' ', s).strip(' \t\r\n.:;|<>-–—')
    return s in NAV_LABELS

def connect(path, readonly=False):
    if readonly:
        db = sqlite3.connect(Path(path).resolve().as_uri() + '?mode=ro', uri=True, timeout=120)
    else:
        db = sqlite3.connect(path, timeout=120, uri=True)
        db.execute('PRAGMA journal_mode=WAL')
        db.execute('PRAGMA synchronous=FULL')
    return db

def metadata(db, key):
    row = db.execute('SELECT value FROM meta WHERE key=?', (key,)).fetchone()
    return json.loads(row[0]) if row else None

def put_meta(db, key, obj):
    db.execute('INSERT OR REPLACE INTO meta VALUES (?,?)', (key,json.dumps(obj,sort_keys=True)))

def normalized_cik(value):
    s = str(value or '').strip()
    if s.endswith('.0'): s = s[:-2]
    return s.zfill(10) if s.isdigit() and len(s)<=10 else ''

def initialize_corpus():
    """One immutable manifest snapshot shared by all three notebooks."""
    manifest = DATA_DIR / 'cache_manifest.jsonl'
    assert manifest.is_file(), f'Missing {manifest}. Copy all_years including cache, not only Excel files.'
    identity = {'manifest_sha256':file_sha(manifest), 'engine':ENGINE_VERSION,
                'remove_exact_navigation':REMOVE_EXACT_NAVIGATION}
    cid = digest(json.dumps(identity, sort_keys=True))[:20]
    root = OUTPUT_ROOT / 'corpora' / cid
    root.mkdir(parents=True, exist_ok=True)
    corpus_path = root / 'corpus.sqlite'
    if not BUILD_CORPUS:
        assert corpus_path.is_file(), 'Run MiniLM Step 3 first to build the shared corpus.'
        db = connect(corpus_path, True)
        assert metadata(db,'complete') is True, 'Shared corpus is not complete. Finish MiniLM Step 3 first.'
        assert metadata(db,'identity') == identity
        db.close()
        return corpus_path, cid
    # This lock is released by the OS after a crash; no manual lock deletion needed.
    with FileLock(str(root/'build.lock'), timeout=0):
        db = connect(corpus_path)
        db.executescript('''
        CREATE TABLE IF NOT EXISTS meta(key TEXT PRIMARY KEY,value TEXT NOT NULL);
        CREATE TABLE IF NOT EXISTS texts(uid INTEGER PRIMARY KEY,text_hash TEXT UNIQUE NOT NULL,text TEXT NOT NULL);
        CREATE TABLE IF NOT EXISTS reports(doc_id TEXT PRIMARY KEY,cik TEXT,fiscal_year TEXT,
          source_path TEXT,n_paragraphs INTEGER,n_sentences INTEGER,n_competition_sentences INTEGER,
          n_navigation_removed INTEGER,review_required INTEGER);
        CREATE TABLE IF NOT EXISTS paragraphs(paragraph_id TEXT PRIMARY KEY,doc_id TEXT NOT NULL,
          ordinal INTEGER,uid INTEGER NOT NULL);
        CREATE INDEX IF NOT EXISTS para_doc ON paragraphs(doc_id,ordinal);
        ''')
        prev = metadata(db,'identity')
        assert prev is None or prev == identity, 'Corpus settings differ; do not reuse this database.'
        with db: put_meta(db,'identity',identity)
        if metadata(db,'complete') is True:
            print('Reusing completed shared corpus:', corpus_path)
            db.close()
            return corpus_path,cid
        # Preserve the exact manifest used; never read the million-row Excel tables.
        snapshot = root/'cache_manifest.snapshot.jsonl'
        if not snapshot.exists():
            import shutil
            shutil.copyfile(manifest, snapshot)
        assert file_sha(snapshot) == identity['manifest_sha256']
        refs = []
        seen = set()
        with snapshot.open(encoding='utf-8-sig') as f:
            for line in f:
                if not line.strip(): continue
                ref = json.loads(line)
                did = ref['doc_id']
                if did in seen: raise ValueError(f'Duplicate report ID: {did}')
                seen.add(did); refs.append(ref)
        done = {x[0] for x in db.execute('SELECT doc_id FROM reports')}
        fallback_index = None
        def resolve_cache(raw):
            nonlocal fallback_index
            s = str(raw).replace('\\','/')
            candidates = [Path(raw)]
            if '/all_years/' in s:
                candidates.append(DATA_DIR / s.rsplit('/all_years/',1)[1])
            if '/cache/' in s:
                candidates.append(DATA_DIR/'cache'/s.rsplit('/cache/',1)[1])
            candidates.extend([DATA_DIR/'cache'/s.rsplit('/',1)[-1],
                               DATA_DIR/'cache'/'relocated'/s.rsplit('/',1)[-1]])
            for p in candidates:
                # Always prefer the copied Windows collection over old absolute paths.
                if p.is_file() and p.resolve().is_relative_to(DATA_DIR.resolve()): return p
            if fallback_index is None:
                fallback_index = {}
                for p in (DATA_DIR/'cache').rglob('*.json.gz'):
                    fallback_index.setdefault(p.name,[]).append(p)
            matches = fallback_index.get(s.rsplit('/',1)[-1],[])
            if len(matches)==1: return matches[0]
            raise FileNotFoundError(f'Cannot uniquely resolve cache: {raw}. Copy/migrate caches first.')
        for ref in tqdm(refs, desc='Build/resume shared paragraph corpus'):
            did = ref['doc_id']
            if did in done: continue
            path = resolve_cache(ref['cache_path'])
            with gzip.open(path,'rt',encoding='utf-8') as f: rec = json.load(f)
            assert rec['filing']['doc_id'] == did
            paras, sents = rec['paragraphs'], rec['sentences']
            assert len(sents)==int(ref['sentence_rows']), f'Sentence count differs: {did}'
            if 'paragraphs' in ref: assert len(paras)==int(ref['paragraphs'])
            ids = [p['paragraph_id'] for p in paras]
            assert len(set(ids))==len(ids)
            bypara = Counter(s['paragraph_id'] for s in sents)
            assert set(bypara) <= set(ids), f'Orphan sentences: {did}'
            kept = [p for p in paras if not (REMOVE_EXACT_NAVIGATION and is_navigation(p['paragraph_text']))]
            assert all(str(p['paragraph_text']).strip() for p in kept), f'Empty paragraph: {did}'
            keepids = {p['paragraph_id'] for p in kept}
            ns = sum(v for k,v in bypara.items() if k in keepids)
            nc = sum(int(s['competition_binary']) for s in sents if s['paragraph_id'] in keepids)
            filing = rec['filing']
            with db:
                for ordinal,p in enumerate(kept):
                    text = p['paragraph_text']  # Hash exact text; never normalize substantive content.
                    th = digest(text)
                    db.execute('INSERT OR IGNORE INTO texts(text_hash,text) VALUES (?,?)',(th,text))
                    uid = db.execute('SELECT uid FROM texts WHERE text_hash=?',(th,)).fetchone()[0]
                    db.execute('INSERT INTO paragraphs VALUES (?,?,?,?)',(p['paragraph_id'],did,ordinal,uid))
                db.execute('INSERT INTO reports VALUES (?,?,?,?,?,?,?,?,?)',
                    (did,normalized_cik(filing.get('cik') or ref.get('cik')),
                     str(filing.get('fiscal_year') or ref.get('fiscal_year') or ''),
                     str(ref.get('source_path','')),len(kept),ns,nc,len(paras)-len(kept),
                     int(bool(filing.get('review_required')))))
        nr = db.execute('SELECT COUNT(*) FROM reports').fetchone()[0]
        assert nr == len(refs), 'Corpus report count does not match manifest.'
        with db:
            put_meta(db,'report_count',nr)
            put_meta(db,'complete',True)
        db.execute('PRAGMA wal_checkpoint(TRUNCATE)'); db.close()
        atomic_json(root/'corpus_receipt.json', {'corpus_id':cid,**identity,'reports':nr,'complete':True})
    return corpus_path,cid

def describe_corpus(path):
    db = connect(path,True)
    n = db.execute('SELECT COUNT(*) FROM texts').fetchone()[0]
    print('Reports:', db.execute('SELECT COUNT(*) FROM reports').fetchone()[0])
    print('Paragraph IDs:',db.execute('SELECT COUNT(*) FROM paragraphs').fetchone()[0])
    print('Unique exact texts:',n)
    print('Navigation paragraphs removed:', db.execute('SELECT SUM(n_navigation_removed) FROM reports').fetchone()[0])
    print('Current vector file size estimate (GiB):',round(n*DIMENSION*2/2**30,2))
    db.close()
    return n

def open_checkpoint(settings, total):
    key = digest(json.dumps(settings,sort_keys=True))[:20]
    folder = OUTPUT_ROOT / MODEL_LABEL / CORPUS_ID / key
    folder.mkdir(parents=True,exist_ok=True)
    db = connect(folder/'checkpoint.sqlite')
    db.executescript('CREATE TABLE IF NOT EXISTS meta(key TEXT PRIMARY KEY,value TEXT NOT NULL);'
                     'CREATE TABLE IF NOT EXISTS done(uid INTEGER PRIMARY KEY,n_tokens INTEGER,n_chunks INTEGER);')
    old = metadata(db,'settings')
    assert old is None or old==settings, 'Checkpoint settings differ.'
    with db: put_meta(db,'settings',settings)
    path = folder/'paragraph_vectors.npy'
    if path.exists():
        vec = np.load(path,mmap_mode='r+')
        assert vec.shape==(total,DIMENSION) and vec.dtype==np.float16
    else:
        assert db.execute('SELECT COUNT(*) FROM done').fetchone()[0]==0, 'Vector file missing. Do not reuse checkpoint.'
        vec = np.lib.format.open_memmap(path,mode='w+',dtype='float16',shape=(total,DIMENSION))
        vec.flush()
    return folder,db,vec

def run_pending(encoder, settings):
    """Flush vector bytes BEFORE marking a batch complete. Recompute uncommitted work on resume."""
    total = N_UNIQUE
    key = digest(json.dumps(settings,sort_keys=True))[:20]
    lockfolder = OUTPUT_ROOT/MODEL_LABEL/CORPUS_ID/key
    lockfolder.mkdir(parents=True,exist_ok=True)
    with FileLock(str(lockfolder/'writer.lock'),timeout=0):
        folder,db,vec = open_checkpoint(settings,total)
        corpus = connect(CORPUS_PATH,True)
        # Attach a read-only corpus; checkpoint is the only writer in this process.
        db.execute('ATTACH DATABASE ? AS corpus',(Path(CORPUS_PATH).resolve().as_uri()+'?mode=ro',))
        # uid is stable and contiguous because corpus is immutable; no INSERT OR IGNORE AUTOINCREMENT gaps.
        assert corpus.execute('SELECT MAX(uid) FROM texts').fetchone()[0]==total
        completed = db.execute('SELECT COUNT(*) FROM done').fetchone()[0]
        print(f'Already embedded: {completed:,}; remaining: {total-completed:,}')
        before = time.perf_counter(); added=0
        last_uid = 0
        with tqdm(total=total,initial=completed,desc=MODEL_LABEL+' unique paragraphs') as bar:
            while True:
                rows = db.execute('SELECT t.uid,t.text FROM corpus.texts t LEFT JOIN done d ON t.uid=d.uid '
                    'WHERE d.uid IS NULL AND t.uid>? ORDER BY t.uid LIMIT ?',
                    (last_uid,PARAGRAPHS_PER_COMMIT)).fetchall()
                if not rows: break
                values,counts,chunks = encoder([r[1] for r in rows])
                values = np.asarray(values,dtype='float32')
                assert values.shape==(len(rows),DIMENSION)
                assert np.isfinite(values).all() and np.all(np.linalg.norm(values,axis=1)>0)
                values /= np.linalg.norm(values,axis=1,keepdims=True)
                positions = np.asarray([r[0]-1 for r in rows],dtype=np.int64)
                vec[positions] = values.astype('float16')
                vec.flush()
                with db:
                    db.executemany('INSERT INTO done VALUES (?,?,?)',
                        [(r[0],int(n),int(c)) for r,n,c in zip(rows,counts,chunks)])
                last_uid=rows[-1][0];added+=len(rows);bar.update(len(rows))
        elapsed=time.perf_counter()-before
        print('New paragraphs / second:',round(added/max(elapsed,.001),2))
        atomic_json(folder/'run_settings.json',settings)
        corpus.close();db.close();del vec
    return folder

def export_results(folder):
    """All report means use each paragraph ID ONCE, including repeated texts in different IDs."""
    settings=json.loads((folder/'run_settings.json').read_text(encoding='utf-8'))
    db=connect(folder/'checkpoint.sqlite',True)
    assert db.execute('SELECT COUNT(*) FROM done').fetchone()[0]==N_UNIQUE, 'Embedding incomplete. Resume Step 4.'
    db.close()
    corpus=connect(CORPUS_PATH,True)
    vectors=np.load(folder/'paragraph_vectors.npy',mmap_mode='r')
    runid=uuid.uuid4().hex[:12]
    index_path=folder/f'paragraph_index_{runid}.csv.gz'
    with gzip.open(index_path,'wt',encoding='utf-8',newline='') as f:
        writer=csv.writer(f)
        writer.writerow(['paragraph_id','doc_id','cik','fiscal_year','text_hash','embedding_row'])
        cursor=corpus.execute('SELECT p.paragraph_id,p.doc_id,r.cik,r.fiscal_year,t.text_hash,p.uid-1 '
          'FROM paragraphs p JOIN reports r ON r.doc_id=p.doc_id JOIN texts t ON t.uid=p.uid ORDER BY p.doc_id,p.ordinal')
        for row in tqdm(cursor,desc='Export paragraph alignment',total=corpus.execute('SELECT COUNT(*) FROM paragraphs').fetchone()[0]):
            writer.writerow(row)
    reports=corpus.execute('SELECT doc_id,cik,fiscal_year,source_path,n_paragraphs,n_sentences,'
       'n_competition_sentences,n_navigation_removed,review_required FROM reports WHERE n_paragraphs>0 ORDER BY doc_id').fetchall()
    report_matrix=folder/f'report_vectors_{runid}.npy'
    rm=np.lib.format.open_memmap(report_matrix,mode='w+',dtype='float32',shape=(len(reports),DIMENSION))
    report_table=folder/f'report_index_{runid}.csv'
    with report_table.open('w',encoding='utf-8-sig',newline='') as f:
        writer=csv.writer(f)
        writer.writerow(['doc_id','cik','fiscal_year','source_path','n_paragraphs','n_sentences',
            'n_competition_sentences','n_navigation_removed','review_required','competition_intensity','embedding_row','corpus_id'])
        for i,r in enumerate(tqdm(reports,desc='Aggregate report embeddings')):
            acc=np.zeros(DIMENSION,dtype='float64');count=0
            cursor=corpus.execute('SELECT uid-1 FROM paragraphs WHERE doc_id=? ORDER BY ordinal',(r[0],))
            while True:
                ids=[x[0] for x in cursor.fetchmany(1024)]
                if not ids: break
                part=np.asarray(vectors[ids],dtype='float32')
                assert np.isfinite(part).all() and np.all(np.linalg.norm(part,axis=1)>0)
                acc += part.sum(axis=0,dtype='float64');count+=len(ids)
            assert count==r[4]
            mean=acc/count;norm=np.linalg.norm(mean)
            assert np.isfinite(norm) and norm>0
            rm[i]=(mean/norm).astype('float32')
            writer.writerow([*r,r[6]/r[5] if r[5] else '',i,CORPUS_ID])
    rm.flush();del rm
    result={'complete':True,'corpus_id':CORPUS_ID,'dimension':DIMENSION,
        'unique_vectors':N_UNIQUE,'paragraph_ids':corpus.execute('SELECT COUNT(*) FROM paragraphs').fetchone()[0],
        'reports_exported':len(reports),'reports_without_paragraphs':corpus.execute('SELECT COUNT(*) FROM reports WHERE n_paragraphs=0').fetchone()[0],
        'report_matrix_sha256':file_sha(report_matrix),'report_index_sha256':file_sha(report_table),
        'paragraph_matrix':'paragraph_vectors.npy','paragraph_dtype':'float16',
        'paragraph_index':index_path.name,'report_matrix':report_matrix.name,'report_index':report_table.name,
        'report_dtype':'float32','report_pooling':'equal weight per paragraph ID, then L2 normalize',
        'corpus_database':str(CORPUS_PATH),'settings':settings,
        'note':'embedding_row is zero-based; join by doc_id or paragraph_id, never by CSV order across models.'}
    atomic_json(folder/f'snapshot_{runid}.json',result)
    atomic_json(folder/'latest_snapshot.json',result)
    corpus.close()
    print(json.dumps(result,indent=2))
    return result

CORPUS_PATH, CORPUS_ID = initialize_corpus()
N_UNIQUE = describe_corpus(CORPUS_PATH)
assert N_UNIQUE > 0
print("Shared corpus ready:", CORPUS_ID)


## Step 4 — Load the model and encode/resume

First use requires internet to download the model. The exact Hugging Face commit is pinned in the output folder and reused on restart. Llama's publisher-provided bidirectional model code is loaded with `trust_remote_code=True` at that pinned commit; MiniLM uses standard Transformers code. GPU memory needs depend on batch size and sequence length; Llama starts at batch 8 and halves automatically after CUDA OOM.

Each chunk uses attention-mask mean pooling, then L2 normalization; chunks are averaged using payload-token counts and the paragraph vector is normalized again. Llama repeats its documented `passage:` prefix for each chunk and saves the first 512 Matryoshka dimensions, normalized. MiniLM saves its native 384 dimensions. All payload tokens are retained; long paragraphs are not silently truncated. These paragraph vectors are an aggregation of chunk representations, not the same as a single unrestricted-context model pass.

The full vector file is preallocated. **Its existence does not mean embedding is complete.** Only the checkpoint and final snapshot establish completion. This step embeds the entire shared corpus.


In [ ]:
def local_run():
    import torch, gc
    from transformers import AutoTokenizer, AutoModel
    from huggingface_hub import model_info
    import importlib.metadata as md
    device=DEVICE_REQUEST
    if device=='auto': device='cuda' if torch.cuda.is_available() else 'cpu'
    if device=='cuda' and not torch.cuda.is_available():
        raise RuntimeError('CUDA unavailable. Install a CUDA PyTorch build in this kernel or set DEVICE_REQUEST="cpu".')
    torch.set_num_threads(CPU_THREADS)
    # Resolve the remote revision once; subsequent resumes use exactly the same commit.
    pin_path=OUTPUT_ROOT/(MODEL_LABEL+'_model_revision.json')
    if pin_path.exists():
        pin=json.loads(pin_path.read_text(encoding='utf-8'))
        assert pin['model']==MODEL_NAME
        revision=pin['revision']
    else:
        revision=model_info(MODEL_NAME).sha
        atomic_json(pin_path,{'model':MODEL_NAME,'revision':revision})
    dtype=torch.float16 if device=='cuda' else torch.float32
    settings={'engine':ENGINE_VERSION,'corpus_id':CORPUS_ID,'model':MODEL_NAME,
        'revision':revision,'dimension':DIMENSION,'sequence_length':MAX_SEQUENCE_LENGTH,
        'pooling':'attention-mask mean per chunk; L2 chunks; payload-token weighted mean; L2 paragraph',
        'chunking':'nonoverlapping token IDs; no truncation; repeat passage prefix per llama chunk',
        'storage':'float16','device':device,'compute_dtype':str(dtype),
        'versions':{x:md.version(x) for x in ['numpy','torch','transformers']}}
    gpu_lock=FileLock(str(OUTPUT_ROOT/'local_gpu.lock'))
    @contextmanager
    def resource():
        if device=='cuda':
            try: gpu_lock.acquire(timeout=0)
            except Exception as e:
                raise RuntimeError('Another embedding notebook holds this GPU. Use MiniLM CPU or finish its run first.') from e
        try: yield
        finally:
            if device=='cuda':gpu_lock.release()
    with resource():
        tok=AutoTokenizer.from_pretrained(MODEL_NAME,revision=revision,trust_remote_code=IS_LLAMA)
        if tok.pad_token_id is None: tok.pad_token=tok.eos_token
        model=AutoModel.from_pretrained(MODEL_NAME,revision=revision,
            trust_remote_code=IS_LLAMA,torch_dtype=dtype,attn_implementation='sdpa').to(device).eval()
        # Infer special-token wrappers from the tokenizer itself; no prepare_for_model dependency.
        probe='embedding tokenizer wrapper probe'
        raw=tok(probe,add_special_tokens=False)['input_ids']
        wrapped=tok(probe,add_special_tokens=True)['input_ids']
        offsets=[i for i in range(len(wrapped)-len(raw)+1) if wrapped[i:i+len(raw)]==raw]
        assert len(offsets)==1, 'Tokenizer has a nonstandard wrapper; do not silently truncate.'
        i=offsets[0];start_tokens=wrapped[:i];end_tokens=wrapped[i+len(raw):]
        prefix=tok('passage: ',add_special_tokens=False)['input_ids'] if IS_LLAMA else []
        limit=MAX_SEQUENCE_LENGTH-len(start_tokens)-len(prefix)-len(end_tokens)
        assert limit>0 and MAX_SEQUENCE_LENGTH<=int(model.config.max_position_embeddings)
        current_batch=[BATCH_SIZE]
        def encode(texts):
            # Each model's own tokenizer covers every token. Sequence windows differ by model.
            sequences=[];owners=[];weights=[];counts=[];nchunks=[]
            for owner,text in enumerate(texts):
                ids=tok(text,add_special_tokens=False,truncation=False,verbose=False)['input_ids']
                assert ids
                counts.append(len(ids));number=0
                for off in range(0,len(ids),limit):
                    part=ids[off:off+limit]
                    sequence=start_tokens+prefix+part+end_tokens
                    assert len(sequence)<=MAX_SEQUENCE_LENGTH
                    sequences.append(sequence);owners.append(owner);weights.append(len(part));number+=1
                nchunks.append(number)
            sums=np.zeros((len(texts),DIMENSION),dtype='float64')
            pos=0
            while pos<len(sequences):
                take=min(current_batch[0],len(sequences)-pos)
                selected=sequences[pos:pos+take]
                longest=max(map(len,selected))
                input_ids=torch.full((take,longest),tok.pad_token_id,dtype=torch.long,device=device)
                mask=torch.zeros((take,longest),dtype=torch.long,device=device)
                for j,seq in enumerate(selected):
                    input_ids[j,:len(seq)]=torch.tensor(seq,device=device)
                    mask[j,:len(seq)]=1
                try:
                    with torch.inference_mode():
                        out=model(input_ids=input_ids,attention_mask=mask,use_cache=False).last_hidden_state if IS_LLAMA else model(input_ids=input_ids,attention_mask=mask).last_hidden_state
                        m=mask.unsqueeze(-1)
                        pooled=(out.float()*m).sum(1)/m.sum(1).clamp(min=1)
                        pooled=pooled[:,:DIMENSION]  # NVIDIA supports Matryoshka dimensions.
                        pooled=torch.nn.functional.normalize(pooled,p=2,dim=1)
                    values=pooled.cpu().numpy()
                except torch.cuda.OutOfMemoryError:
                    del input_ids,mask
                    if 'out' in locals():del out
                    if 'pooled' in locals():del pooled
                    torch.cuda.empty_cache()
                    if take<=1:raise RuntimeError('GPU OOM at batch 1. Reduce MAX_SEQUENCE_LENGTH (new checkpoint), or use CPU.')
                    current_batch[0]=max(1,take//2)
                    print('Reducing GPU batch size to',current_batch[0]);continue
                for j,value in enumerate(values):
                    sums[owners[pos+j]]+=value*weights[pos+j]
                pos+=take
                del out,pooled,input_ids,mask
            norms=np.linalg.norm(sums,axis=1,keepdims=True)
            assert np.isfinite(sums).all() and np.all(norms>0)
            return (sums/norms).astype('float32'),counts,nchunks
        try:
            v,_,_=encode(['Competition in this business is intense.'])
            assert v.shape==(1,DIMENSION) and np.isfinite(v).all()
            print('Model check passed:',v.shape,'| device:',device,'| revision:',revision)
            folder=run_pending(encode,settings)
        finally:
            del model
            gc.collect()
            if device=='cuda':torch.cuda.empty_cache()
    return folder

MODEL_OUTPUT_DIR = local_run()
print("Embedding checkpoint:", MODEL_OUTPUT_DIR)


## Step 5 — Export paragraph alignment and report-level GENI inputs

Run after Step 4 finishes. Completion requires every unique text to have a committed vector. It exports:

* `paragraph_vectors.npy`: all unique paragraph vectors, float16, one stable row per exact text.
* `paragraph_index_<snapshot>.csv.gz`: every paragraph ID → report ID/CIK/fiscal-year label/text hash/vector row.
* `report_vectors_<snapshot>.npy`: equal-weight mean of each report's paragraph IDs, L2 normalized, float32.
* `report_index_<snapshot>.csv`: report ID, original fiscal-year label, source, counts, review flag, cleaned competition intensity and vector row.
* `latest_snapshot.json`: exact filenames/settings for this model and this corpus. Published only after export completes.

Use each paragraph ID once in the report mean. Identical paragraphs with distinct IDs still count as distinct retained paragraphs; deduplication saves inference, not statistical weight. Reports without retained paragraphs are counted in the snapshot and excluded from the report matrix. The report-level inputs must later be joined to financial outcomes by `doc_id` using the verified financial-period link; the original extracted `fiscal_year` is not a substitute for period validation.


In [ ]:
SNAPSHOT = export_results(MODEL_OUTPUT_DIR)
print("COMPLETE:", MODEL_LABEL, "| corpus:", CORPUS_ID, "| reports:", SNAPSHOT["reports_exported"])


## Use the report embeddings later

The financial table should contain one row per `doc_id`. Join it to the report index with `validate="one_to_one"`, then select matrix rows using that model's `embedding_row`. Compare all three models on the intersection of eligible `doc_id`s with identical T, Y, W, folds and exclusions. Embedding dimension can differ; it does not change the observational unit. Do not concatenate the three matrices and call it a model comparison.

For chunk/window or dimension sensitivity, use a new configuration/checkpoint. Choose specifications using predeclared diagnostics and held-out tasks, not by which one produces a significant causal estimate. These embeddings neither validate Item 1 boundaries nor establish causal identification.

Source documentation: [MiniLM](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2), [NVIDIA Llama embedding model](https://huggingface.co/nvidia/llama-nemotron-embed-1b-v2), [OpenAI embeddings](https://developers.openai.com/api/docs/guides/embeddings), [OpenAI pricing](https://developers.openai.com/api/docs/models/text-embedding-3-small).


In [ ]:
# Example only: edit the financial table path, then uncomment.
# import pandas as pd
# snap = json.loads((MODEL_OUTPUT_DIR / "latest_snapshot.json").read_text())
# index = pd.read_csv(MODEL_OUTPUT_DIR / snap["report_index"], dtype={"doc_id":str,"cik":str})
# H = np.load(MODEL_OUTPUT_DIR / snap["report_matrix"], mmap_mode="r")
# finance = pd.read_csv(r"data/restricted/verified_financial_outcomes.csv", dtype={"doc_id":str})
# assert not finance["doc_id"].duplicated().any()
# joined = finance.merge(index, on="doc_id", how="inner", validate="one_to_one", suffixes=("_finance",""))
# H_joined = H[joined["embedding_row"].to_numpy(dtype=int)]
# print(joined.shape, H_joined.shape)
